# B2-020-language-transformers — Practice p22 — Solution

**Type:** scenario · **Difficulty:** intro · **Concepts:** transformer-nlp-task-design

*55 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260812`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C11-neural-training`, `B2-019-attention-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb).

## Solution

| Deployment | Output object / axes | Visibility | Head | Loss | Primary metric |
|---|---|---|---|---|---|
| Intent classification | one `(B,C)` vector per sequence, from a pooled non-padding representation | bidirectional | `Linear(8,C)` | sequence cross-entropy | macro-F1, because rare intents matter |
| Named-entity tagging | `(B,N,C)` token labels | bidirectional | token-wise `Linear(8,C)` | cross-entropy over non-padding tokens | entity-span F1, matching boundary/type errors |
| Left-to-right command generation | `(B,N,V)` next-token distributions | causal | vocabulary head | shifted token cross-entropy over non-padding targets | exact command match, because one wrong token can invalidate execution |
| Query-document retrieval | one vector per query/document, then pair scores `(B,K)` | bidirectional | projection plus similarity | contrastive ranking loss | Recall@K, matching whether a useful document is retrieved |

Padding is excluded from pooling, token loss, and metrics. Classification/retrieval collapse the sequence axis; tagging/generation retain it.

In [ ]:
DESIGNS = {
    "intent": ("sequence", "bidirectional", "linear classifier", "cross entropy", "macro-F1"),
    "ner": ("token", "bidirectional", "token classifier", "masked cross entropy", "entity-span F1"),
    "generation": ("token", "causal", "vocabulary head", "shifted cross entropy", "exact command match"),
    "retrieval": ("sequence pair", "bidirectional", "projection and similarity", "contrastive ranking", "Recall@K"),
}

### Answer check

In [ ]:
assert DESIGNS["generation"][1] == "causal"
assert DESIGNS["ner"][0] == "token"
assert DESIGNS["intent"][4] == "macro-F1"
assert DESIGNS["retrieval"][4] == "Recall@K"